In [ ]:
# Day 1: inspect sources before training; never infer video order from SMOKE5K filenames.
from pathlib import Path
import json, collections, torch, zipfile
root=Path('/kaggle/input')
report={'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'sources':[str(p) for p in root.iterdir()],'checkpoints':[],'manifests':[],'archives':[]}
for p in root.rglob('best.pth'):
 c=torch.load(p,map_location='cpu',weights_only=True)
 report['checkpoints'].append({'path':str(p),'update':c.get('update'),'validation':c.get('validation'),'n_tensors':len(c.get('state_dict',{}))})
 del c
for p in root.rglob('manifest.json'):
 m=json.loads(p.read_text())
 report['manifests'].append({'path':str(p),'splits':{k:len(v) for k,v in m.items() if isinstance(v,list)},'examples':{k:v[:2] for k,v in m.items() if isinstance(v,list)}})
for p in root.rglob('*.zip'):
 with zipfile.ZipFile(p) as z:
  ns=z.namelist(); report['archives'].append({'path':str(p),'n_files':len(ns),'examples':ns[:12]})
report['temporal_status']='UNVERIFIED: sequence, camera and timestamp metadata required before temporal training.'
Path('/kaggle/working/day1_audit.json').write_text(json.dumps(report,indent=2,default=str))
print('DAY1_AUDIT',json.dumps(report,default=str),flush=True)


In [ ]:
import torch, subprocess
print('GPU_CHECK',torch.__version__,torch.cuda.is_available())
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)


In [ ]:
# Obtain the author's SmokeSeg archive and inspect labels before any temporal training.
from pathlib import Path
import subprocess,sys,json,zipfile,time,requests,re
from bs4 import BeautifulSoup
work=Path('/kaggle/working'); start=time.time()
print('DATA_PREP_START',flush=True)
subprocess.run([sys.executable,'-m','pip','-q','install','gdown'],check=True)
import gdown
archive=work/'SmokeSeg.zip'
if not archive.is_file():
 gdown.download(id='1AIHZ5PckjHGqeaG3VY5WZgH-Y-5lAzA0',output=str(archive),quiet=True)
assert archive.is_file()
with zipfile.ZipFile(archive) as z:
 names=z.namelist()
 print('SMOKESEG_ARCHIVE',len(names),names[:35],flush=True)
 textfiles=[n for n in names if n.lower().endswith(('.txt','.csv','.json'))]
 info={'archive_bytes':archive.stat().st_size,'n_files':len(names),'text_files':textfiles,'sample_names':names[:100],'metadata_samples':{n:z.read(n).decode('utf8',errors='replace')[:1200] for n in textfiles[:8]}}
 (work/'smokeseg_archive_audit.json').write_text(json.dumps(info,indent=2))
 print('SMOKESEG_METADATA',json.dumps(info),flush=True)
 for n in names:
  assert (work/'SmokeSeg-data'/n).resolve().is_relative_to((work/'SmokeSeg-data').resolve())
 z.extractall(work/'SmokeSeg-data')
print('DATA_PREP_DONE_SECONDS',time.time()-start,flush=True)


In [ ]:
# Audit real sequence identity; exclude any event whose frames were seen during SMOKE5K pretraining.
from pathlib import Path
from PIL import Image
import numpy as np,re,json,collections,hashlib
r=Path('/kaggle/working/SmokeSeg-data/SmokeSeg')
ims={p.stem:p for p in (r/'JPEGImages').iterdir() if p.suffix.lower() in ('.jpg','.png','.jpeg') and not p.name.startswith('.')}
masks={p.stem:p for p in (r/'SegmentationClass').iterdir() if p.suffix=='.png' and not p.name.startswith('.')}
cp=next(Path('/kaggle/input').rglob('fosp_smoke5k_14000/manifest.json')); old=json.loads(cp.read_text())
seen={x['id'] for k in ('train','val','test') for x in old[k]}
rows=[]; excluded=set(); values=collections.Counter()
for key,p in ims.items():
 if key not in masks or '__' not in key: continue
 seq,stem=key.rsplit('__',1); m=re.fullmatch(r'(\d{10})_([+-]\d+)',stem)
 if not m: continue
 event=seq.split('_')[0]
 if stem in seen: excluded.add(event)
 rows.append(dict(id=key,sequence=seq,event=event,timestamp=int(m[1]),offset=int(m[2]),image=str(p),mask=str(masks[key])))
for x in rows[:20]:
 with Image.open(x['mask']) as im: values.update(map(int,np.unique(np.asarray(im))))
seqs=collections.defaultdict(list)
for x in rows:
 if x['event'] not in excluded: seqs[x['sequence']].append(x)
pairs={'train':[],'val':[],'test':[]}
for seq,items in sorted(seqs.items()):
 items.sort(key=lambda x:x['timestamp']); event=items[0]['event']; h=int(hashlib.sha256(('42:'+event).encode()).hexdigest()[:8],16)%10
 split='test' if h==0 else 'val' if h==1 else 'train'
 for a,b in zip(items,items[1:]):
  if 0<b['timestamp']-a['timestamp']<=300:
   pairs[split].append(dict(**b,previous=a['image'],delta_seconds=b['timestamp']-a['timestamp']))
summary={'images':len(ims),'masks':len(masks),'parsed':len(rows),'excluded_pretraining_dates':sorted(excluded),'mask_values_sample':dict(values),'pairs':{k:len(v) for k,v in pairs.items()},'events':{k:len({x['event'] for x in v}) for k,v in pairs.items()},'limitation':'date grouping conservative; source IDs exclusion covers known IDs, not all visual near-duplicates; no negative frames yet'}
Path('/kaggle/working/temporal_manifest.json').write_text(json.dumps(pairs,indent=2))
Path('/kaggle/working/temporal_audit.json').write_text(json.dumps(summary,indent=2))
print('TEMPORAL_AUDIT',json.dumps(summary),flush=True)
print('EXAMPLES',json.dumps({k:v[:1] for k,v in pairs.items()}),flush=True)


In [ ]:
from pathlib import Path
import sys,os,subprocess,json
root=Path('/kaggle/input')
repo=next(root.rglob('FoSp-official/config/fosp_config.py')).parent.parent
mmcv=Path('/kaggle/temp/mmcv')
if not (mmcv/'mmcv').is_dir(): subprocess.run(['git','clone','--depth','1','--branch','v1.6.2','https://github.com/open-mmlab/mmcv.git',str(mmcv)],check=True)
subprocess.run([sys.executable,'-m','pip','-q','install','addict','yapf==0.40.1','easydict','timm==0.6.13','regex','setuptools<81'],check=True)
deps=next(root.rglob('fosp-deps/timm/__init__.py')).parent.parent
sys.path[:0]=[str(repo),str(repo/'inpainter_setting'),str(mmcv),str(deps)]
os.environ['PYTHONPATH']=os.pathsep.join(sys.path)
import torch,numpy as np
from mmcv import Config
from mmseg.models import build_segmentor
from inpainter_setting.torch_utils.ops import bias_act,upfirdn2d
from torch_utils.ops import bias_act as ba,upfirdn2d as uf
for op in (bias_act,upfirdn2d,ba,uf): op._init=lambda:False
cfg=Config.fromfile(str(repo/'config/fosp_config.py')); cfg.model.pretrained=None
model=build_segmentor(cfg.model).cuda().eval()
weights=next(root.rglob('fosp_smoke5k_14000/best.pth'))
c=torch.load(weights,map_location='cpu',weights_only=True); model.load_state_dict(c['state_dict'],strict=True); del c
print('FOSP_LOADED',torch.cuda.get_device_name(0),str(weights),flush=True)


In [ ]:
# Day-1 pilot: frozen FoSp + equal-capacity static/temporal residual heads.
# This is NOT the full learned background-memory method and NOT the official SmokeSeg protocol.
import time,random,json,shutil,traceback
from pathlib import Path
import numpy as np,torch
import torch.nn.functional as F
from PIL import Image
out=Path('/kaggle/working/day1_pilot');out.mkdir(exist_ok=True)
pairs=json.loads(Path('/kaggle/working/temporal_manifest.json').read_text())
assert all(pairs[k] for k in ('train','val','test'))
assert not ({x['event'] for x in pairs['train']} & {x['event'] for x in pairs['val']+pairs['test']})
SEED=42;random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.set_num_threads(2)
model.eval().requires_grad_(False)
class ResidualHead(torch.nn.Module):
 def __init__(self):
  super().__init__();self.net=torch.nn.Sequential(torch.nn.Conv2d(9,24,3,padding=1),torch.nn.ReLU(),torch.nn.Conv2d(24,24,3,padding=1),torch.nn.ReLU(),torch.nn.Conv2d(24,1,1))
  torch.nn.init.zeros_(self.net[-1].weight);torch.nn.init.zeros_(self.net[-1].bias)
 def forward(self,x):return x[:,:1]+self.net(x)
heads={k:ResidualHead().cuda() for k in ('static','temporal')}
heads['temporal'].load_state_dict(heads['static'].state_dict())
opts={k:torch.optim.AdamW(h.parameters(),lr=1e-4,weight_decay=1e-4) for k,h in heads.items()}
mean=np.array([123.675,116.28,103.53],np.float32);std=np.array([58.395,57.12,57.375],np.float32)
cache={};cache_limit=3200
@torch.no_grad()
def image_features(path):
 if path in cache:return tuple(t.cuda().float() for t in cache[path])
 with Image.open(path) as im: rgb=np.asarray(im.convert('RGB').resize((512,512),Image.Resampling.BILINEAR),dtype=np.float32)
 x=torch.from_numpy(((rgb-mean)/std).transpose(2,0,1).copy())[None].cuda()
 state=np.random.get_state();np.random.seed(123)
 try: logit=model.encode_decode(x,[{}])
 finally:np.random.set_state(state)
 logit=F.interpolate(logit,size=(128,128),mode='bilinear',align_corners=False).clamp(-15,15)
 rgbsmall=F.interpolate(torch.from_numpy(rgb.transpose(2,0,1).copy())[None].cuda()/255,size=(128,128),mode='bilinear',align_corners=False)
 if len(cache)<cache_limit:cache[path]=(logit.cpu().half(),rgbsmall.cpu().half())
 return logit,rgbsmall
@torch.no_grad()
def features(row,flip=False):
 cur,rgb=image_features(row['image']);old,prev=image_features(row['previous'])
 diff=(rgb-prev).abs(); rel=torch.exp(-4*diff.mean(1,keepdim=True))
 temporal=torch.cat((cur,old,rgb,diff,rel),1)
 static=torch.cat((cur,torch.zeros_like(old),rgb,torch.zeros_like(diff),torch.zeros_like(rel)),1)
 with Image.open(row['mask']) as im:
  y=torch.from_numpy((np.asarray(im.convert('L').resize((128,128),Image.Resampling.NEAREST))>0).astype(np.float32))[None,None].cuda()
 if flip:static=static.flip(-1);temporal=temporal.flip(-1);y=y.flip(-1)
 return {'static':static,'temporal':temporal},y
@torch.no_grad()
def evaluate():
 sums={k:[] for k in ('fosp','static','temporal')}
 for i,row in enumerate(pairs['val']):
  xs,y=features(row); truth=y.bool(); logits={'fosp':xs['static'][:,:1],**{k:h(xs[k]) for k,h in heads.items()}}
  for k,z in logits.items():
   p=z.sigmoid()>0.4;tp=(p&truth).sum().item();fp=(p&~truth).sum().item();fn=(~p&truth).sum().item()
   sums[k].append([2*tp/max(2*tp+fp+fn,1),tp/max(tp+fp+fn,1)])
  if (i+1)%100==0:print('VAL_PROGRESS',i+1,len(pairs['val']),flush=True)
 return {k:{'f1':float(np.mean(v,axis=0)[0]),'smoke_iou':float(np.mean(v,axis=0)[1])} for k,v in sums.items()}
def save(payload,path):
 temp=path.with_suffix('.tmp');torch.save(payload,temp);temp.replace(path)
config={'method':'frozen FoSp + residual correction pilot','input_size':512,'head_and_eval_size':128,'threshold':0.4,'seed':42,'budget_seconds':10800,'max_updates':6000,'eval_every':500,'patience':5,'min_delta':0.001,'test_used':False,'limitations':['positive pairs only; cannot measure fog false alarms','coarse 128px pilot metrics not comparable to native-resolution SMOKE5K scores','camera overlap possible across dates; not unseen-camera evaluation','reliability is a fixed heuristic, not a learned memory controller']}
(out/'config.json').write_text(json.dumps(config,indent=2))
shutil.copy2('/kaggle/working/temporal_manifest.json',out/'manifest.json');shutil.copy2('/kaggle/working/temporal_audit.json',out/'data_audit.json')
started=time.monotonic();update=0;best={k:-1. for k in heads};history=[];stale=0
if (out/'last.pth').is_file():
 c=torch.load(out/'last.pth',map_location='cpu',weights_only=False)
 for k in heads:heads[k].load_state_dict(c['heads'][k]);opts[k].load_state_dict(c['optimizers'][k])
 update=c['update'];best=c['best'];history=c['history'];random.setstate(c['python_rng']);np.random.set_state(c['numpy_rng']);torch.set_rng_state(c['torch_rng']);torch.cuda.set_rng_state_all(c['cuda_rng']);stale=c.get('stale',0)
 print('RESUMED',update,flush=True)
else:
 initial=evaluate();history.append({'update':0,'validation':initial});print('PILOT_INITIAL',json.dumps(initial),flush=True)
 for k in heads:
  best[k]=initial[k]['f1'];save({'state_dict':heads[k].state_dict(),'update':0,'validation':initial[k],'config':config},out/('best_'+k+'.pth'))
def snapshot():
 save({'update':update,'heads':{k:h.state_dict() for k,h in heads.items()},'optimizers':{k:o.state_dict() for k,o in opts.items()},'best':best,'history':history,'stale':stale,'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all(),'config':config},out/'last.pth')
 (out/'history.json').write_text(json.dumps(history,indent=2))
 (out/'status.json').write_text(json.dumps({'update':update,'elapsed_seconds':time.monotonic()-started,'best':best},indent=2))
try:
 while update<6000 and time.monotonic()-started<10200:
  row=random.choice(pairs['train']);xs,y=features(row,random.random()<0.5);losses={}
  for k,h in heads.items():
   opts[k].zero_grad(set_to_none=True);z=h(xs[k]);p=z.sigmoid();dice=1-(2*(p*y).sum()+1)/(p.sum()+y.sum()+1);loss=F.binary_cross_entropy_with_logits(z,y)+dice;loss.backward();opts[k].step();losses[k]=float(loss.detach())
  update+=1
  if update%25==0: print('PILOT_TRAIN',json.dumps({'update':update,'loss':losses,'elapsed':round(time.monotonic()-started,1),'cache':len(cache)}),flush=True)
  if update%100==0:snapshot()
  if update%500==0:
   scores=evaluate();history.append({'update':update,'validation':scores});improved=False
   for k in heads:
    if scores[k]['f1']>best[k]:
     improved |= scores[k]['f1']>best[k]+0.001;best[k]=scores[k]['f1'];save({'state_dict':heads[k].state_dict(),'update':update,'validation':scores[k],'config':config},out/('best_'+k+'.pth'))
   stale=0 if improved else stale+1;snapshot();print('PILOT_VALIDATION',json.dumps(history[-1]),flush=True)
   if stale>=5:print('EARLY_STOP',stale,flush=True);break
finally:
 snapshot();shutil.copy2(weights,out/'fosp_base_14000.pth');shutil.make_archive('/kaggle/working/forestwatch_day1_pilot','zip',out)
 print('PILOT_SAVED',update,'/kaggle/working/forestwatch_day1_pilot.zip',flush=True)
